In [1]:
import json

# 파일 경로 설정 (실제 파일 경로로 변경 필요)
file_path = 'kmi.json'

def inspect_kmi_structure(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        print(f"✅ 데이터 로드 성공! 총 {len(data)}개의 샘플이 있습니다.\n")
        
        # 첫 번째 샘플의 구조 출력
        print("🔍 [첫 번째 샘플 구조 미리보기]")
        print(json.dumps(data[0], indent=2, ensure_ascii=False))
        
        # 키(Key) 확인
        print(f"\n🔑 최상위 키(Keys): {list(data[0].keys())}")
        
        # Dialogue 내부 구조 확인
        if 'dialogue' in data[0]:
            print(f"🔑 대화(Dialogue) 샘플 키: {list(data[0]['dialogue'][0].keys())}")
            
    except Exception as e:
        print(f"❌ 오류 발생: {e}")

# 실행
inspect_kmi_structure(file_path)

✅ 데이터 로드 성공! 총 1000개의 샘플이 있습니다.

🔍 [첫 번째 샘플 구조 미리보기]
{
  "id": 1,
  "category_ko": "정신건강",
  "category_en": "Mental Health",
  "dialogue": [
    {
      "role": "Therapist",
      "utterance_ko": "안녕하세요, 최근에 어떤 생각이나 감정이 반복되었나요?",
      "utterance_en": "Hi, what thoughts or feelings have you been having recently that keep coming up for you?",
      "label": "Open Question"
    },
    {
      "role": "Client",
      "utterance_ko": "다른 사람들이 열정적으로 살아가는 것을 보며 저도 저렇게 해야 한다고 생각하지만, 의욕이 들지 않아요.",
      "utterance_en": "I see other people living their lives with passion and I think I should be doing that, but I don't feel motivated."
    },
    {
      "role": "Therapist",
      "utterance_ko": "타인의 열정적인 모습을 보며 무엇을 해야 할지 고민하셨군요.",
      "utterance_en": "You were wondering what to do while watching someone else's passion.",
      "label": "General"
    },
    {
      "role": "Client",
      "utterance_ko": "네, 그래서 항상 제 자신을 탓하게 되고, 저도 모르게 자책감에 빠지곤 해요.",
      "utterance_en": "Yes, so I always en

In [ ]:
import json
import os
import random
import time
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv  # .env 파일 로드용

# ==========================================
# 0. 환경 변수 로드 및 설정
# ==========================================
# .env 파일에서 환경 변수 불러오기
load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY")

# API 키 확인 (없으면 에러 발생시켜서 실수 방지)
if not API_KEY:
    raise ValueError("❌ .env 파일에서 OPENAI_API_KEY를 찾을 수 없습니다. 확인해주세요.")

client = OpenAI(api_key=API_KEY)

# ==========================================
# 1. 설정 (Configuration)
# ==========================================
INPUT_FILE = "kmi.json"
OUTPUT_FILE = "dpo_train_data.jsonl"
TARGET_SAMPLE_COUNT = 2000
  # 예산을 위해 일단 500개(x2 = 1000쌍)만 생성
MODEL_NAME = "gpt-4o-mini" # 가성비 모델

# ==========================================
# 2. 프롬프트 템플릿 (Prompt Template)
# ==========================================
SYSTEM_PROMPT = """
당신은 '동기 강화 상담(Motivational Interviewing)' 전문 임상 심리학자 AI입니다.
주어진 상담 대화 맥락(History)을 분석하여 내담자(Client)의 페르소나를 파악하고,
지시된 '상담 단계(Session Phase)'에 맞춰 두 가지 버전의 내담자 답변을 생성해야 합니다.
반드시 JSON 형식으로 출력하세요.
"""

USER_PROMPT_TEMPLATE = """
### 1. Dialogue Context (대화 맥락)
- 카테고리: {category}
- 대화 내역:
{history_text}

### 2. Current Turn (현재 상담사 질문)
- 상담사: "{therapist_query}"

### 3. Generation Task
위 맥락을 바탕으로, 아래 **[설정된 상황]**에 맞는 내담자의 답변 두 가지(Winning vs Losing)를 생성하시오.

**[설정된 상황: {phase_description}]**

**A. Winning Response (Best)**
- {winning_guide}
- 말투: 한국어 구어체, 문맥 자연스러움.

**B. Losing Response (Worst)**
- {losing_guide}

### 4. Output Format (JSON Only)
{{
  "phase": "{phase_name}",
  "therapist_query": "{therapist_query}",
  "chosen": "Winning 답변 텍스트",
  "rejected": "Losing 답변 텍스트",
  "rationale": "임상적 선택 이유"
}}
"""

# ==========================================
# 3. 데이터 생성 로직
# ==========================================

def generate_dpo_pair(history, therapist_query, category, phase_type):
    """
    phase_type: 'early' (저항) or 'late' (변화)
    """
    if phase_type == 'early':
        phase_name = "Session 1 (초기 저항 단계)"
        # KMI 논문 등에서 언급된 초기 저항의 특징 반영 (단답형, 방어적 태도) [cite: 42]
        phase_desc = "상담 초기이며, 내담자는 아직 상담사를 신뢰하지 않고 자신의 문제 직면을 회피하거나 저항(Resistance)하는 상태입니다."
        winning_guide = "방어적이고, 짧으며, 문제의 원인을 외부로 돌리거나 상담의 효과를 의심하는 태도를 보이세요."
        losing_guide = "너무 협조적이거나, 처음부터 자신의 잘못을 인정하고 적극적으로 해결하려는 비현실적인 태도."
    else: # late
        phase_name = "Session 4+ (변화 및 통찰 단계)"
        # KMI 논문의 Change Talk (DARN) 반영 [cite: 297, 589]
        phase_desc = "상담이 진행되어 라포가 형성되었습니다. 내담자는 '변화 대화(Change Talk)'를 시작하며 문제 해결 의지(Desire, Ability, Reason, Need)를 보입니다."
        winning_guide = "상담사를 신뢰하며, 자신의 문제에 대해 통찰(Insight)하고 변화를 시도하려는 의지를 구체적으로 표현하세요."
        losing_guide = "여전히 초기처럼 방어적이거나, 혹은 문맥과 전혀 상관없는 엉뚱한 이야기를 하는 환각(Hallucination) 반응."

    prompt = USER_PROMPT_TEMPLATE.format(
        category=category,
        history_text=history,
        therapist_query=therapist_query,
        phase_description=phase_desc,
        phase_name=phase_name,
        winning_guide=winning_guide,
        losing_guide=losing_guide
    )

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.7
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Error generating: {e}")
        return None

# ==========================================
# 4. 메인 실행 루프
# ==========================================
def main():
    # KMI 원본 데이터 로드
    if not os.path.exists(INPUT_FILE):
        print(f"❌ {INPUT_FILE} 파일을 찾을 수 없습니다.")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    
    print(f"총 {len(raw_data)}개의 원본 대화 로드 완료.")
    
    # 예산 관리를 위해 랜덤 샘플링
    if len(raw_data) > TARGET_SAMPLE_COUNT // 2:
        target_indices = random.sample(range(len(raw_data)), TARGET_SAMPLE_COUNT // 2)
    else:
        target_indices = range(len(raw_data))
    
    generated_count = 0
    cost_estimate = 0.0
    
    print(f"🚀 데이터 생성 시작... (목표: 약 {TARGET_SAMPLE_COUNT}개 쌍)")

    with open(OUTPUT_FILE, 'w', encoding='utf-8') as out_f:
        for idx in tqdm(target_indices, desc="Generating DPO Data"):
            dialogue = raw_data[idx].get('dialogue', [])
            category = raw_data[idx].get('category_ko', 'General')
            
            # 대화가 너무 짧으면 스킵
            if len(dialogue) < 4: continue
            
            # 상담사(Therapist)의 발화 턴 찾기
            therapist_indices = [i for i, turn in enumerate(dialogue) if turn['role'] == 'Therapist']
            if not therapist_indices: continue
            
            # 중간 쯤에 있는 질문 하나 선택 (너무 초반이나 끝은 피함)
            target_t_idx = random.choice(therapist_indices[1:-1]) if len(therapist_indices) > 2 else therapist_indices[0]
            
            # Context 구성 (해당 질문 직전까지의 대화)
            history_lines = []
            for i in range(target_t_idx):
                role = "상담사" if dialogue[i]['role'] == 'Therapist' else "내담자"
                text = dialogue[i]['utterance_ko']
                history_lines.append(f"{role}: {text}")
            
            # 최근 6턴만 사용하여 토큰 절약 [cite: 256]
            history_text = "\n".join(history_lines[-6:]) 
            therapist_query = dialogue[target_t_idx]['utterance_ko']
            
            # 1. Early Phase Pair 생성 (저항 단계)
            early_data = generate_dpo_pair(history_text, therapist_query, category, 'early')
            if early_data is None or 'phase' not in early_data:
                print("❌ Early Phase 데이터 생성 실패. 건너뜁니다.")
                continue
            dpo_entry = {
                "prompt": f"[상황: {early_data['phase']}]\n대화 내역:\n{history_text}\n상담사: {therapist_query}\n내담자:",
                "chosen": early_data['chosen'],
                "rejected": early_data['rejected'],
                "source": "synthetic_kmi_early"
            }            
            out_f.write(json.dumps(dpo_entry, ensure_ascii=False) + "\n")
            generated_count += 1

            # 2. Late Phase Pair 생성 (변화 단계)
            late_data = generate_dpo_pair(history_text, therapist_query, category, 'late')
            if late_data and 'phase' in late_data:
                dpo_entry = {
                    "prompt": f"[상황: {late_data['phase']}]\n대화 내역:\n{history_text}\n상담사: {therapist_query}\n내담자:",
                    "chosen": late_data['chosen'],
                    "rejected": late_data['rejected'],
                    "source": "synthetic_kmi_late"
                }
                out_f.write(json.dumps(dpo_entry, ensure_ascii=False) + "\n")
                generated_count += 1
                
            # 대략적인 비용 계산 (gpt-4o-mini 기준)
            cost_estimate += 0.0006

    print(f"\n🎉 생성 완료! 총 {generated_count}개의 DPO 쌍이 {OUTPUT_FILE}에 저장되었습니다.")
    print(f"💰 추정 소요 비용: 약 ${cost_estimate:.4f}")

if __name__ == "__main__":
    main()

총 1000개의 원본 대화 로드 완료.
🚀 데이터 생성 시작... (목표: 약 2000개 쌍)


Generating DPO Data: 100%|██████████| 1000/1000 [2:46:53<00:00, 10.01s/it] 


🎉 생성 완료! 총 2000개의 DPO 쌍이 dpo_train_data.jsonl에 저장되었습니다.
💰 추정 소요 비용: 약 $0.6000


In [5]:
import json
import os

# ==========================================
# 설정
# ==========================================
INPUT_FILE = "kmi.json"
OUTPUT_FILE = "sft_train_data.jsonl"

def main():
    # 파일 존재 확인
    if not os.path.exists(INPUT_FILE):
        print(f"❌ 오류: '{INPUT_FILE}' 파일을 찾을 수 없습니다.")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    
    sft_data = []
    total_turns = 0
    
    print(f"📂 원본 데이터 {len(raw_data)}개 로드 완료.")

    for sample in raw_data:
        dialogue = sample.get('dialogue', [])
        # KMI 데이터 구조에 따라 category 키 확인 (category_ko 등)
        category = sample.get('category_ko', sample.get('category', '일반 상담'))
        
        history = []
        
        for turn in dialogue:
            # 역할 매핑 (Therapist -> 상담사, Client -> 내담자)
            role = "상담사" if turn['role'] == 'Therapist' else "내담자"
            text = turn.get('utterance_ko', turn.get('text', ''))
            
            # 내담자(Client)의 발화를 학습 목표(Output)로 설정
            if turn['role'] == 'Client':
                # 문맥(History)이 있는 경우에만 데이터 생성 (첫 턴 제외 가능)
                if len(history) > 0:
                    # 최근 6턴 정도만 프롬프트에 포함하여 문맥 유지 (Context Window 관리)
                    context_window = history[-6:]
                    context_text = "\n".join(context_window)
                    
                    # 직전 턴이 상담사인지 확인 (일반적으로 그렇지만 예외 처리)
                    last_turn = history[-1]
                    if last_turn.startswith("상담사:"):
                        current_query = last_turn.replace("상담사: ", "").strip()
                        
                        # 데이터 포맷 구성 (Alpaca 스타일)
                        entry = {
                            "instruction": f"당신은 {category} 문제로 상담을 받고 있는 내담자입니다. 상담사의 질문에 대해 당신의 상황과 감정에 맞게 자연스럽게 대답하세요.",
                            "input": f"---대화 문맥---\n{context_text}\n\n---현재 질문---\n상담사: {current_query}",
                            "output": text
                        }
                        sft_data.append(entry)
            
            # History 업데이트 (다음 턴을 위해 현재 발화 추가)
            history.append(f"{role}: {text}")
            total_turns += 1

    # 결과 저장
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for entry in sft_data:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
            
    print(f"✅ 변환 완료! 총 {len(sft_data)}개의 학습 샘플이 '{OUTPUT_FILE}'에 저장되었습니다.")

if __name__ == "__main__":
    main()

📂 원본 데이터 1000개 로드 완료.
✅ 변환 완료! 총 8558개의 학습 샘플이 'sft_train_data.jsonl'에 저장되었습니다.


#   🚀 1. SFT 학습 코드 (train_sft.py)

In [ ]:
# =================================================================
# File: train_sft.py
# Description: KMI 데이터를 활용한 내담자 페르소나 기초 학습 (SFT)
# =================================================================

from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
from huggingface_hub import login
from dotenv import load_dotenv
import torch
import os

# 1. 환경 설정 및 HF 로그인
# ---------------------------------------------------------
load_dotenv()
WRITE_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_WRITE_ONLY")
READ_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_READ_ONLY")

if not WRITE_TOKEN:
    raise ValueError("❌ .env에서 Write Token을 찾을 수 없습니다.")

# Hugging Face 로그인 (모델 업로드를 위해 필수)
login(token=WRITE_TOKEN)

# 설정 변수
MODEL_NAME = "MLP-KTLim/llama-3-Korean-Bllossom-8B" # 한국어 Base Model
DATA_FILE = "sft_train_data.jsonl"
OUTPUT_DIR = "sft_lora_model"
HF_REPO_ID = "YOUR_HF_ID/Patient-AI-SFT-v1" # [수정 필요] 본인 HF ID 입력

MAX_SEQ_LENGTH = 2048
DTYPE = None # Auto detection
LOAD_IN_4BIT = True # 메모리 절약

# 2. 모델 및 토크나이저 로드
# ---------------------------------------------------------
print(f"🚀 SFT 학습을 위해 모델 로드 중: {MODEL_NAME}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
    token = READ_TOKEN # 읽기 토큰 사용
)

# 3. LoRA 어댑터 설정
# ---------------------------------------------------------
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# 4. 데이터셋 준비
# ---------------------------------------------------------
dataset = load_dataset("json", data_files=DATA_FILE, split="train")

# 프롬프트 포맷팅 (Alpaca Style)
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

# 5. 학습 설정 (Trainer)
# ---------------------------------------------------------
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # 데이터가 적으므로 1 epoch 권장
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "sft_checkpoints",
    ),
)

# 6. 학습 실행 및 저장
# ---------------------------------------------------------
print("🔥 SFT 학습 시작...")
trainer.train()

print(f"💾 로컬 저장 중: {OUTPUT_DIR}")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Hugging Face 업로드 (선택 사항이지만 권장)
print(f"☁️ Hugging Face 업로드 중: {HF_REPO_ID}")
try:
    model.push_to_hub(HF_REPO_ID, token=WRITE_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=WRITE_TOKEN)
    print("✅ SFT 모델 업로드 완료!")
except Exception as e:
    print(f"⚠️ 업로드 실패 (로컬 파일은 저장됨): {e}")

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
from huggingface_hub import login
from dotenv import load_dotenv
import torch
import os

# 1. 환경 설정
load_dotenv()
WRITE_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_WRITE_ONLY")
READ_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_READ_ONLY")
login(token=WRITE_TOKEN)

# 70B 모델 설정 (H200 전용)
MODEL_NAME = "MLP-KTLim/llama-3-Korean-Bllossom-70B"
OUTPUT_DIR = "sft_model_a"
HF_REPO_ID = "YOUR_HF_ID/Patient-AI-Model-A-SFT-70B" # [수정 필요]

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# 2. 모델 로드
print(f"🚀 Loading Model: {MODEL_NAME}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
    token = READ_TOKEN
)

# 3. LoRA 설정 (70B 모델에 맞춰 Rank 조정 가능하나 16이면 충분)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 4. 데이터셋
dataset = load_dataset("json", data_files="sft_train_data.jsonl", split="train")
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    texts = [alpaca_prompt.format(i, inp, out) + tokenizer.eos_token for i, inp, out in zip(examples["instruction"], examples["input"], examples["output"])]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

# 5. 학습 설정 (H200 최적화)
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = dataset, dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH, dataset_num_proc = 4, packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 16, # H200의 힘! 배치를 키움
        gradient_accumulation_steps = 1,  # 배치가 크니 누적은 줄여서 속도 향상
        warmup_ratio = 0.1,
        num_train_epochs = 1, 
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "sft_checkpoints",
    ),
)

# 6. 실행
print("🔥 SFT Start...")
trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
try:
    model.push_to_hub(HF_REPO_ID, token=WRITE_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=WRITE_TOKEN)
    print("✅ Model A (SFT) Uploaded!")
except: pass

# 🚀 2. DPO 학습 코드 (train_dpo.py)

In [ ]:
# =================================================================
# File: train_dpo.py
# Description: SFT 모델 기반 DPO 학습 (저항/변화 단계 최적화)
# =================================================================

from unsloth import FastLanguageModel, PatchDPOTrainer
from unsloth import is_bfloat16_supported
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset
from huggingface_hub import login
from dotenv import load_dotenv
import torch
import os

# 1. 환경 설정 및 HF 로그인
# ---------------------------------------------------------
load_dotenv()
WRITE_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_WRITE_ONLY")
READ_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_READ_ONLY")

login(token=WRITE_TOKEN)

# 설정 변수
# 🚨 중요: 반드시 SFT 학습이 끝난 로컬 폴더를 지정해야 함
SFT_MODEL_PATH = "sft_lora_model" 
DPO_DATA_FILE = "dpo_train_data.jsonl"
OUTPUT_DIR = "final_dpo_model"
HF_REPO_ID = "YOUR_HF_ID/Patient-AI-DPO-v1" # [수정 필요] 본인 HF ID 입력

MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

# 2. SFT 모델 로드
# ---------------------------------------------------------
print(f"🚀 DPO 학습을 위해 SFT 완료된 모델 로드 중: {SFT_MODEL_PATH}")

# Unsloth는 Adapter 경로를 입력하면 Base Model과 자동으로 병합하여 로드함
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = SFT_MODEL_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

# 3. DPO용 LoRA 어댑터 추가 설정
# ---------------------------------------------------------
# 기존 SFT 어댑터 위에 DPO 학습을 위한 설정을 덮어씌움 (계속 학습)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# 4. 데이터셋 로드
# ---------------------------------------------------------
dataset = load_dataset("json", data_files=DPO_DATA_FILE, split="train")

# 5. DPO Trainer 설정
# ---------------------------------------------------------
PatchDPOTrainer() # Unsloth 메모리 최적화 패치

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # Unsloth는 Reference Model 없이 학습 가능 (메모리 절약 핵심)
    tokenizer = tokenizer,
    beta = 0.1, # DPO 온도 (Preference 강도)
    train_dataset = dataset,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 1, 
        learning_rate = 5e-6, # 🚨 DPO는 SFT보다 낮은 Learning Rate 필수
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.05,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "dpo_checkpoints",
    ),
)

# 6. 학습 실행 및 저장
# ---------------------------------------------------------
print("🔥 DPO 학습 시작... (시간이 좀 걸릴 수 있음)")
dpo_trainer.train()

print(f"💾 최종 모델 로컬 저장 중: {OUTPUT_DIR}")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Hugging Face 업로드
print(f"☁️ Hugging Face 업로드 중: {HF_REPO_ID}")
try:
    model.push_to_hub(HF_REPO_ID, token=WRITE_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=WRITE_TOKEN)
    print("🎉 모든 프로젝트 학습 완료! 수고했네 연구원!")
except Exception as e:
    print(f"⚠️ 업로드 실패: {e}")

# 최종평가 

In [ ]:
import os
import json
import torch
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from unsloth import FastLanguageModel
from dotenv import load_dotenv

# ==========================================
# 1. 환경 설정 & 상수 정의
# ==========================================
load_dotenv()

# API 키 로드
OPENAI_CLIENT = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
HF_READ_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_READ_ONLY")

# 평가할 모델 ID (Hugging Face Hub 경로 또는 로컬 경로)
# 예: "YourID/Patient-AI-SFT-v1"
SFT_MODEL_ID = "sft_lora_model"   # 혹은 HF ID 입력
DPO_MODEL_ID = "final_dpo_model"  # 혹은 HF ID 입력

# 평가자 모델 (Judge)
JUDGE_MODEL = "gpt-4o-mini"

# 테스트 케이스 (논문의 'Session Phase' 개념 반영)
TEST_CASES = [
    # Case A: 초기 저항 (Session 1)
    {
        "phase": "Session 1 (초기 저항)",
        "persona": "내담자는 우울증 초기이며, 상담에 비협조적이고 자신의 문제 원인을 외부 환경 탓으로 돌리는 경향이 있음(Resistance).",
        "query": "안녕하세요, 오늘 상담을 오시게 된 특별한 계기가 있으신가요?"
    },
    {
        "phase": "Session 1 (초기 저항)",
        "persona": "내담자는 무기력증이 심하며, 상담사가 자신을 이해하지 못할 것이라 생각하여 방어적인 태도를 보임.",
        "query": "요즘 수면 패턴은 좀 어떠신가요?"
    },
    # Case B: 후기 변화 (Session 5)
    {
        "phase": "Session 5 (변화/통찰)",
        "persona": "내담자는 상담사와 라포가 형성되었으며, 자신의 우울감이 생각 습관에서 비롯됨을 깨닫고 변화하려는 의지(Change Talk)를 보임.",
        "query": "지난주에 이야기했던 '하루 10분 산책하기'는 시도해 보셨나요?"
    },
    {
        "phase": "Session 5 (변화/통찰)",
        "persona": "내담자는 이제 자신의 강점을 찾고 구체적인 미래 계획을 세우려 함(Ability/Plan).",
        "query": "앞으로 어떤 모습으로 살아가고 싶으신가요?"
    }
]

# ==========================================
# 2. LLM-as-a-Judge 함수 (논문 메트릭 구현)
# ==========================================
JUDGE_PROMPT_TEMPLATE = """
당신은 대화형 AI의 '페르소나 일관성(Persona Consistency)'을 평가하는 엄격한 임상 심리 판사(Judge)입니다.
아래 제공된 [페르소나]와 AI의 [답변]을 비교하여, 답변이 설정된 성격, 상황, 상담 단계(Phase)와 일치하는지 평가하세요.

### 평가 기준 (Prompt-to-Line Consistency) [cite: 220, 222]
1. **Resistance (초기):** 저항 단계일 경우, 너무 협조적이거나 밝은 태도는 감점입니다.
2. **Change Talk (후기):** 변화 단계일 경우, 구체적인 변화 의지나 통찰이 없으면 감점입니다.
3. **Hallucination:** 페르소나에 없는 내용을 지어내거나 문맥에 맞지 않는 말은 1점입니다.

### 입력 정보
- [페르소나/상황]: {persona} (단계: {phase})
- [상담사 질문]: {query}
- [AI 답변]: {response}

### 출력 형식 (JSON)
점수는 1점(완전 불일치)에서 5점(완벽 일치) 사이의 정수로 매기십시오.
{{
  "score": 점수,
  "reason": "점수 부여 사유 (1문장)"
}}
"""

def evaluate_consistency(phase, persona, query, response):
    try:
        prompt = JUDGE_PROMPT_TEMPLATE.format(
            phase=phase, persona=persona, query=query, response=response
        )
        
        completion = OPENAI_CLIENT.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": "JSON 포맷으로만 응답하세요."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0
        )
        return json.loads(completion.choices[0].message.content)
    except Exception as e:
        print(f"❌ Eval Error: {e}")
        return {"score": 0, "reason": "Error"}

# ==========================================
# 3. 추론(Inference) 함수
# ==========================================
def run_inference_for_model(model_path, test_cases):
    print(f"\n🚀 모델 로드 중: {model_path}")
    
    # Unsloth 모델 로드 (HF Read Token 사용)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
        token=HF_READ_TOKEN 
    )
    FastLanguageModel.for_inference(model)
    
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
당신은 상담을 받고 있는 내담자입니다. 주어진 상황과 단계에 맞춰 자연스럽게 대답하세요.

### Input:
---상황 설정---
{persona} (단계: {phase})

---상담사 질문---
{query}

### Response:
"""
    
    responses = []
    for case in tqdm(test_cases, desc="Generating Responses"):
        prompt = alpaca_prompt.format(
            phase=case['phase'],
            persona=case['persona'],
            query=case['query']
        )
        
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        
        # 답변 추출
        response_text = decoded.split("### Response:\n")[-1].strip()
        responses.append(response_text)
    
    # 메모리 정리를 위해 모델 삭제
    del model, tokenizer
    torch.cuda.empty_cache()
    
    return responses

# ==========================================
# 4. 메인 실행 로직
# ==========================================
def main():
    # 1) SFT 모델 추론
    print("--- [Stage 1] SFT Model Inference ---")
    sft_responses = run_inference_for_model(SFT_MODEL_ID, TEST_CASES)
    
    # 2) DPO 모델 추론
    print("--- [Stage 2] DPO Model Inference ---")
    dpo_responses = run_inference_for_model(DPO_MODEL_ID, TEST_CASES)
    
    # 3) LLM-as-a-Judge 평가
    print("\n--- [Stage 3] LLM-as-a-Judge Evaluation ---")
    results = []
    
    for i, case in enumerate(tqdm(TEST_CASES, desc="Judging")):
        # SFT 평가
        sft_eval = evaluate_consistency(case['phase'], case['persona'], case['query'], sft_responses[i])
        
        # DPO 평가
        dpo_eval = evaluate_consistency(case['phase'], case['persona'], case['query'], dpo_responses[i])
        
        results.append({
            "Phase": case['phase'],
            "Query": case['query'],
            "Persona": case['persona'],
            "SFT_Response": sft_responses[i],
            "SFT_Score": sft_eval['score'],
            "SFT_Reason": sft_eval['reason'],
            "DPO_Response": dpo_responses[i],
            "DPO_Score": dpo_eval['score'],
            "DPO_Reason": dpo_eval['reason']
        })

    # 4) 결과 집계 및 저장
    df = pd.DataFrame(results)
    
    avg_sft = df['SFT_Score'].mean()
    avg_dpo = df['DPO_Score'].mean()
    
    print("\n📊 [최종 평가 결과]")
    print("="*50)
    print(f"✅ SFT 모델 평균 점수: {avg_sft:.2f} / 5.0")
    print(f"✅ DPO 모델 평균 점수: {avg_dpo:.2f} / 5.0")
    print("="*50)
    
    if avg_dpo > avg_sft:
        print("🎉 성공! DPO 모델이 SFT 모델보다 페르소나 일관성이 더 뛰어납니다.")
    else:
        print("⚠️ 주의: DPO 효과가 미미합니다. 데이터셋이나 하이퍼파라미터를 점검하세요.")
        
    df.to_csv("final_model_evaluation.csv", index=False, encoding="utf-8-sig")
    print("\n💾 상세 결과가 'final_model_evaluation.csv'로 저장되었습니다.")

if __name__ == "__main__":
    main()